# C3: quét layer x kích thước hộp trên dev (Qwen2.5-VL-3B)

Chỉ chạy trên **dev**. Chọn cấu hình dựa trên dev rồi mới chạy test một lần.
Notebook tự ghi đè `vicrop_relatt.py` bản mới, nên **không cần upload lại dataset**. Cần dataset `da2-dev` (bản cũ vẫn dùng được), GPU T4 x2, Internet On.

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils pillow numpy
import os, sys, shutil, glob

hits = glob.glob('/kaggle/input/**/vicrop_relatt.py', recursive=True)
assert hits, 'Không thấy vicrop_relatt.py: kiểm tra đã Add Input dataset da2-dev chưa'
src_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
print('Dataset root:', src_root)
shutil.copytree(src_root, '/kaggle/working/da2', dirs_exist_ok=True)
os.chdir('/kaggle/working/da2'); sys.path.insert(0, '.')

import torch
print(torch.cuda.device_count(), 'GPU:', torch.cuda.get_device_name(0))

In [ ]:
%%writefile src/crop/vicrop_relatt.py
"""ViCrop `rel_att` cho Qwen2.5-VL (tự cài lại, nhánh C3).

Bám cơ chế của saccharomycetes/mllms_know (qwen2_5_methods.py, utils.py), nhưng
được cài lại từ phần mô tả code gốc, chưa đối chiếu từng dòng. Cần diff với
bản gốc trước khi báo cáo; mọi chỗ khác gốc phải ghi vào báo cáo.

Cơ chế:
  1. Forward 2 lần cùng ảnh: prompt câu hỏi và prompt chung.
  2. Lấy attention của token cuối tới các token ảnh ở layer ATT_LAYER, trung bình theo head.
  3. rel_att = att_câu_hỏi / att_prompt_chung (khử thiên lệch vị trí).
  4. Reshape về lưới (h/2, w/2), tìm cửa sổ có tổng attention cao nhất ở nhiều tỉ lệ.
"""

import numpy as np

ATT_LAYER = 22  # mặc định của repo gốc
RATIOS = (1, 1.2, 1.4, 1.6, 1.8, 2)
GENERAL_PROMPT = "Write a general description of the image."  # [cần đối chiếu với repo gốc]


def bbox_from_att_map(att_map: np.ndarray, image_size: tuple, bbox_size: int):
    """Chọn hộp vuông quanh vùng attention cao nhất.

    att_map: (H_grid, W_grid). image_size: (W, H) pixel. bbox_size: cạnh hộp cơ sở (pixel).
    Trả về (x1, y1, x2, y2) pixel, nằm trong ảnh.
    """
    W, H = image_size
    gh, gw = att_map.shape
    bw, bh = W / gw, H / gh  # kích thước 1 ô lưới theo pixel

    best = None  # (độ nhọn, tâm_x, tâm_y, cạnh)
    for r in RATIOS:
        side = bbox_size * r
        nx, ny = max(1, round(side / bw)), max(1, round(side / bh))
        nx, ny = min(nx, gw), min(ny, gh)

        # tổng attention của mọi cửa sổ nx x ny, qua ảnh tích phân
        ii = np.pad(att_map, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
        sums = ii[ny:, nx:] - ii[:-ny, nx:] - ii[ny:, :-nx] + ii[:-ny, :-nx]
        y, x = np.unravel_index(np.argmax(sums), sums.shape)
        peak = sums[y, x]

        # độ nhọn: đỉnh so với trung bình các cửa sổ lân cận, chuẩn hóa theo diện tích
        neigh = [sums[j, i] for j in (y - 1, y, y + 1) for i in (x - 1, x, x + 1)
                 if (j, i) != (y, x) and 0 <= j < sums.shape[0] and 0 <= i < sums.shape[1]]
        sharp = (peak - (np.mean(neigh) if neigh else 0.0)) / (nx * ny)
        if best is None or sharp > best[0]:
            best = (sharp, (x + nx / 2) * bw, (y + ny / 2) * bh, side)

    _, cx, cy, side = best
    side = min(side, W, H)
    cx = min(max(cx, side / 2), W - side / 2)
    cy = min(max(cy, side / 2), H - side / 2)
    return (int(cx - side / 2), int(cy - side / 2), int(cx + side / 2), int(cy + side / 2))


def rel_att_map(model, processor, image, question_prompt: str, layer: int = ATT_LAYER):
    """Trả về rel_att (H_grid, W_grid) của một layer."""
    return rel_att_maps(model, processor, image, question_prompt, (layer,))[layer]


def rel_att_maps(model, processor, image, question_prompt: str, layers=(ATT_LAYER,)):
    """Trả về {layer: rel_att (H_grid, W_grid)}. Một lần forward cho mọi layer.

    Chạy trên GPU, cần attn_implementation='eager'.
    """
    import torch

    def run(prompt):
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model(**inp, output_attentions=True)
        ids = inp["input_ids"][0]
        tok = processor.tokenizer
        pos = (ids == tok.convert_tokens_to_ids("<|vision_start|>")).nonzero()[0].item() + 1
        end = (ids == tok.convert_tokens_to_ids("<|vision_end|>")).nonzero()[0].item()
        grid = (inp["image_grid_thw"][0, 1:] // 2).tolist()
        return {l: out.attentions[l][0, :, -1, pos:end].mean(dim=0).float().cpu().numpy().reshape(grid)
                for l in layers}

    q, g = run(question_prompt), run(GENERAL_PROMPT)
    return {l: q[l] / (g[l] + 1e-8) for l in layers}


In [ ]:
import json, time, random, unicodedata
import numpy as np
from pathlib import Path
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from src.crop.vicrop_relatt import rel_att_maps, bbox_from_att_map, GENERAL_PROMPT

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CFG = dict(
    model='Qwen/Qwen2.5-VL-3B-Instruct', dtype='float16', attn='eager',
    max_pixels=1024 * 28 * 28, layers=[14, 22, 30], bbox_sizes=[336, 512, 768, 1024],
    general_prompt=GENERAL_PROMPT,
    answer_suffix='\nTrả lời ngắn gọn bằng một cụm từ hoặc một câu.',
    max_new_tokens=64, split='dev',
)
processor = AutoProcessor.from_pretrained(CFG['model'], max_pixels=CFG['max_pixels'])
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CFG['model'], torch_dtype=torch.float16, device_map='auto', attn_implementation='eager').eval()
print('Số layer của model:', getattr(model.config, 'text_config', model.config).num_hidden_layers)

In [ ]:
def answer(image, question):
    msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': question + CFG['answer_suffix']}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = processor(text=[text], images=[image], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=CFG['max_new_tokens'], do_sample=False)
    res = processor.batch_decode(out[:, inp['input_ids'].shape[1]:], skip_special_tokens=True)[0]
    return unicodedata.normalize('NFC', res.strip())

dev = [json.loads(l) for l in open('data/processed/visr_bench_vi/qa_vi_haiku_dev.jsonl', encoding='utf-8') if l.strip()]
out_dir = Path('/kaggle/working/C3_vicrop') if os.path.isdir('/kaggle/working') else Path('results/C3_vicrop')
out_dir.mkdir(parents=True, exist_ok=True)
json.dump(CFG, open(out_dir / 'dev_sweep_config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
fout = open(out_dir / 'dev_sweep.jsonl', 'w', encoding='utf-8')

for n, it in enumerate(dev, 1):
    img = Image.open(f"data/raw/visr_bench_vi/{it['image']}").convert('RGB')
    q = unicodedata.normalize('NFC', it['question'])
    maps = rel_att_maps(model, processor, img, q + CFG['answer_suffix'], layers=CFG['layers'])
    for layer in CFG['layers']:
        for bs in CFG['bbox_sizes']:
            box = bbox_from_att_map(maps[layer], img.size, bs)
            row = dict(qid=it['qid'], gold=it['answer'], layer=layer, bbox_size=bs, bbox=box,
                       image_size=img.size, pred_crop=answer(img.crop(box), q))
            fout.write(json.dumps(row, ensure_ascii=False) + '\n'); fout.flush()
    print(f'{n}/{len(dev)}', it['qid'])

fout.close()
print('\nĐã lưu tại', out_dir)
print('=== dev_sweep.jsonl ===')
print(open(out_dir / 'dev_sweep.jsonl', encoding='utf-8').read())